#### ch 7.1 지시 미세 튜닝 소개  
LLM을 사전 훈련하는 것은 한 번에 한 단어 씩 생성하는 법을 배우는 것이다. 이렇게 만들어진 사전 훈련된 LLM은 이 텍스트 완성 능력이 있다.  
즉, 텍스트의 일부를 입력 받아 문장을 완성하거나 문단 전체를 작성할 수 있다. 하지만 사전 훈련된 LLM은 '이 텍스트의 문법을 고쳐 줘' 또는  
'이 텍스트를 수동태로 바꿔 줘' 같은 구체적인 명령을 잘 수행하지 못한다. 나중에 instrucition fine-tuning 또는 supervised instruction fine-tuning을 위해  
사전 훈련된 LLM을 로드하여 사용하는 구체적인 예제를 살펴보자!  
지시를 따르고 기대하는 응답을 생성하도록 LLM의 능력을 향상시키는 데 중점을 둔다. 지시 미세 튜닝의 핵심 요소는 데이터셋 준비이다.  
이번에는 데이터셋 준비부터 지시 미세 튜닝의 세 단계에 걸친 모든 과정을 수행해보자!


In [45]:
import json
import os
import urllib

def download_and_load_file(file_path, url):
    if not os.path.exists(file_path):
        with urllib.request.urlopen(url) as response:
            text_data = response.read().decode("utf-8")
        with open(file_path, "w", encoding="utf-8") as file:
            file.write(text_data)
            
    with open(file_path, "r") as file:
        data = json.load(file)
        
    return data

file_path ="instruction-data.json"
url = (
    "https://raw.githubusercontent.com/rickiepark/llm-from-scratch/main/"
    "ch07/01_main-chapter-code/instruction-data.json"
)

data = download_and_load_file(file_path, url)
print("샘플 개수: ", len(data))

샘플 개수:  1100


In [46]:
print("샘플 예시: \n", data[50])

샘플 예시: 
 {'instruction': 'Identify the correct spelling of the following word.', 'input': 'Ocassion', 'output': "The correct spelling is 'Occasion.'"}


In [47]:
print("다른 샘플: \n", data[999])

다른 샘플: 
 {'instruction': "What is an antonym of 'complicated'?", 'input': '', 'output': "An antonym of 'complicated' is 'simple'."}


지시 미세 튜닝은 JSON 파일에서 추출한 샘플처럼 입력-출력 쌍으로 구성된 데이터셋에서 모델을 훈련한다.  
LLM을 위해 샘플을 포맷팅하는 방법은 여러가지가 있는데 그 중 유명한 LLM인 Alpaca와 Phi-3를 훈련하는데 사용된 2가지 포맷이 있다.  
이를 종종 프롬프트 스타일이라고 부른다.  
알파카는 초기 LLM 중 하나로 지시 미세 튜닝 과정에 대한 내용이 공개되어 있다. MicroSoft에서 개발한 Phi-3는 프롬프트 스타일의 다양성을 보여 주기 위해  
예로 들었다. 여기서는 나머지 부분에서는 알파카 프롬프트 스타일을 사용한다. 왜냐면 알파카 스타일이 초기 미세 튜닝 방법을 정의하는데 크게 기여를 했고,  
인기가 많은 포맷이기 때문이다.

In [48]:
def format_input(entry):
    instruction_text = (
        f"Below is an instruction that describe a task. "
        f"Write a response that appropriately completes the request."
        f"\n\n### Instruction:\n{entry['instruction']}"
    )
    
    input_text = (
        f"\n\n### Input:\n{entry['input']}" if entry["input"] else ""
    )
    
    return instruction_text + input_text

In [49]:
model_input = format_input(data[50])
desired_response = f"\n\n### Response: \n{data[50]['output']}"
print(model_input + desired_response)

Below is an instruction that describe a task. Write a response that appropriately completes the request.

### Instruction:
Identify the correct spelling of the following word.

### Input:
Ocassion

### Response: 
The correct spelling is 'Occasion.'


In [50]:
model_input = format_input(data[999])
desired_response = f"\n\n### Response: \n{data[999]['output']}"
print(model_input + desired_response)

Below is an instruction that describe a task. Write a response that appropriately completes the request.

### Instruction:
What is an antonym of 'complicated'?

### Response: 
An antonym of 'complicated' is 'simple'.


이제 파이토치 데이터 로더를 만들기 전에 이전에 했던 것처럼 스팸 분류 데이터셋으로 했던 것 처럼 데이터셋을 훈련 세트, 검증 세트, 테스트 세트로 나누자!

In [51]:
train_portion = int(len(data) * 0.85) # 훈련에 전체 데이터의 85% 사용한다는 것
test_portion = int(len(data) * 0.1) # 테스트에 10% 사용
val_portion = len(data) - test_portion - train_portion

train_data = data[:train_portion]
test_data = data[train_portion:train_portion+test_portion]
val_data = data[train_portion+test_portion:]

print("훈련 세트 크기: ", len(train_data))
print("검증 세트 크기: ", len(val_data))
print("테스트 세트 크기: ", len(test_data))

훈련 세트 크기:  935
검증 세트 크기:  55
테스트 세트 크기:  110


#### 7.3 훈련 배치 만들기  
지시 미세 튜닝 과정의 구현 단계의 다음 스텝은 훈련 배치를 효과적으로 구성하는 것이다.  
이를 통해 미세 튜닝 과정에서 모델에게 포맷팅된 훈련 배치를 제공할 수 있다.  
이전 장에서 pytorch DataLoader 클래스로 훈련 배치를 자동으로 만들었다. 이 클래스는 샘플 리스트를 배치로 묶어 주는 기본 콜레이트 함수를 사용한다.  
콜레이트 함수는 훈련하는 동안 개별 데이터 샘플의 리스트를 하나로 합쳐 모델이 효과적으로 처리할 수 있도록 한다
하지만 지시 미세 튜닝을 위한 배치 구성은 조금 더 복잡하기 때문에 DataLoader에 적용할 사용자 정의 콜레이트 함수를 정의해야 한다.  
이 콜레이트 함수를 구현하여 지시 미세 튜닝 데이터셋에서 요구되는 작업과 포맷팅을 처리하자!  
사용자 정의 콜레이트 함수 구현을 포함하여 배치 처리 과정을 몇 단계로 나눠 수행해보자!  
일단 모든 샘플에 format_input 함수를 적용하고, 토큰화하는 InstructionDataset 클래스를 구현한다.  
이 클래스는 앞선 6장의 SpamDataset 클래스와 매우 비슷하다.  

배치 과정을 구현하기 위한 처음 두 단계. 먼저 특정 프롬프트 템플릿으로 샘플을 포맷팅하고(2.1), 그런 다음 토큰화를 한다.  
이를 통해 모델이 처리할 수 있는 토큰 ID의 시퀀스를 만든다.

In [52]:
import sys
import torch

print(sys.executable)
print(torch.__version__)
print(torch.nn.Module)

c:\Users\dhson\miniconda3\envs\llm-from-scratch\python.exe
2.6.0+cu124
<class 'torch.nn.modules.module.Module'>


In [53]:
import torch
from torch.utils.data import Dataset

class InstructionDataset(Dataset):
    def __init__(self, data, tokenizer):
        self.data = data
        self.encoded_texts = []
        for entry in data:
            instruction_plus_input = format_input(entry)
            response_text = f"\n\n### Response:\n{entry['output']}"
            full_text = instruction_plus_input + response_text
            self.encoded_texts.append(
                tokenizer.encode(full_text)
            )
            
    def __getitem__(self, index):
        return self.encoded_texts[index]
    
    def __len__(self):
        return len(self.data)

분류 미세 튜닝에서 사용했던 방식과 비슷하게 여러 개의 훈련 샘플을 배치로 묶어서 훈련 속도를 높이는 게 좋다!  
이렇게 하려면 모든 입력의 길이가 같도록 패딩을 추가해야 한다. 분류 미세 튜닝에서처럼 <|endoftext|> 토큰을 패딩 토큰으로 사용한다.  
텍스트 입력에 <|endoftext|> 토큰을 패팅 토큰으로 사용한다. 텍스트 입력에 <|endoftext|> 토큰을 추가하는 대신에 <|endoftext|>에 해당하는 토큰 ID를 토큰화된 입력에 바로 추가할 수 있다.  
토크나이저의 .encode 메서드를 사용해 <|endoftext|> 토큰 ID를 확인해보자!

In [54]:
import tiktoken
tokenizer = tiktoken.get_encoding("gpt2")
print(tokenizer.encode("<|endoftext|>", allowed_special={"<|endoftext|>"}))

[50256]


더 복잡한 작업을 수행하기 위해 데이터 로더에 전달할 사용자 정의 콜레이트 함수를 작성해보자! 이 콜레이트 함수는 배치에 있는 훈련 샘플의 길이가 동일하도록  
패딩을 추가한다. 물론 배치마다 길이가 다를 수 있다. 이 방식은 전체 데이터셋이 아니라 각 배치에서 가장 긴 샘플에 맞춰 시퀀스를 확장하므로 불피요한 패딩을  
최소화한다. 이제 아래에서 사용자 정의 콜레이트 함수로 패딩 처리를 구현해보자!

In [55]:
def custom_collate_draft_1(batch, pad_token_id=50256, device="cpu"):
    batch_max_length = max(len(item)+1 for item in batch) # 배치에서 가장 긴 시퀀스를 찾는다.
    inputs_lst = []
    
    for item in batch:
        new_item = item.copy()
        new_item += [pad_token_id]

        padded = (
            new_item + [pad_token_id] * (batch_max_length - len(new_item))
        )
        inputs = torch.tensor(padded[:-1]) # 이전에 추가로 넣은 패딩 토큰을 제외한다.
        inputs_lst.append(inputs)
        
    inputs_tensor = torch.stack(inputs_lst).to(device)
    return inputs_tensor
    

In [56]:
inputs_1 = [0, 1, 2, 3, 4]
inputs_2 = [5, 6]
inputs_3 = [7, 8, 9]
batch = (
    inputs_1,
    inputs_2,
    inputs_3
)
print(custom_collate_draft_1(batch))

tensor([[    0,     1,     2,     3,     4],
        [    5,     6, 50256, 50256, 50256],
        [    7,     8,     9, 50256, 50256]])


In [57]:
def custom_collate_draft_2(batch, pad_token_id=50256, device="cpu"):
    batch_max_length = max(len(item)+1 for item in batch)
    inputs_lst, targets_lst = [], []
    
    for item in batch:
        new_item = item.copy()
        new_item += [pad_token_id]
        
        padded = (
            new_item + [pad_token_id] * (batch_max_length - len(new_item))
        )
        
        inputs = torch.tensor(padded[:-1])
        targets = torch.tensor(padded[1:])
        inputs_lst.append(inputs)
        targets_lst.append(targets)
        
    inputs_tensor = torch.stack(inputs_lst).to(device)
    targets_tensor = torch.stack(targets_lst).to(device)
    
    return inputs_tensor, targets_tensor


inputs, targets = custom_collate_draft_2(batch)
print(inputs)
print(targets) 

tensor([[    0,     1,     2,     3,     4],
        [    5,     6, 50256, 50256, 50256],
        [    7,     8,     9, 50256, 50256]])
tensor([[    1,     2,     3,     4, 50256],
        [    6, 50256, 50256, 50256, 50256],
        [    8,     9, 50256, 50256, 50256]])


다음 코드는 사용자 콜레이트 함수를 수정하여 타깃 리스트에서 ID가 50256인 토큰을 -100으로 바꾼다. 또한 allowed_max_length 매개변수를 추가하여 선택적으로  
샘플의 길이를 제한하도록 만든다. 사용하려는 데이터셋에 GPT-2 모델이 지원하는 1,024 토큰의 문맥 길이보다 긴 샘플이 있을 때 유용하다.

In [58]:
def custom_collate_fn(
    batch, 
    pad_token_id = 50256,
    ignore_index = -100,
    allowed_max_length = None,
    device = "cpu"):
    batch_max_length = max(len(item)+1 for item in batch)
    inputs_lst, targets_lst = [], []
    
    for item in batch:
        new_item = item.copy()
        new_item += [pad_token_id]
        
        padded = (
            new_item + [pad_token_id] * (batch_max_length - len(new_item))
        )
        inputs = torch.tensor(padded[:-1])
        targets = torch.tensor(padded[1:])
        mask = targets == pad_token_id
        indices = torch.nonzero(mask).squeeze()
        if indices.numel() > 1:
            targets[indices[1:]] = ignore_index
            
        if allowed_max_length is not None:
            inputs = inputs[:allowed_max_length]
            targets = targets[:allowed_max_length]
            
        inputs_lst.append(inputs)
        targets_lst.append(targets)
        
    inputs_tensor = torch.stack(inputs_lst).to(device)
    targets_tensor = torch.stack(targets_lst).to(device)
    return inputs_tensor, targets_tensor

In [59]:
inputs, targets = custom_collate_fn(batch)
print(inputs)
print(targets)

tensor([[    0,     1,     2,     3,     4],
        [    5,     6, 50256, 50256, 50256],
        [    7,     8,     9, 50256, 50256]])
tensor([[    1,     2,     3,     4, 50256],
        [    6, 50256,  -100,  -100,  -100],
        [    8,     9, 50256,  -100,  -100]])


In [60]:
logits_1 = torch.tensor(
    [[-1.0, 1.0],
     [-0.5, 1.5]]
)

targets_1 = torch.tensor([0,1])
loss_1 = torch.nn.functional.cross_entropy(logits_1, targets_1)
print(loss_1)

tensor(1.1269)


In [61]:
logits_2 = torch.tensor(
    [[-1.0, 1.0],
     [-0.5, 1.5],
     [-0.5, 1.5]]
)
targets_2 = torch.tensor([0, 1, 1])
loss_2 = torch.nn.functional.cross_entropy(logits_2, targets_2)
print(loss_2)

tensor(0.7936)


In [62]:
targets_3 = torch.tensor([0,1,-100])
loss_3 = torch.nn.functional.cross_entropy(logits_2, targets_3)
print(loss_3)
print("loss_1 == loss_3: ",loss_1 == loss_3)

tensor(1.1269)
loss_1 == loss_3:  tensor(True)


위에서 loss_1과 loss_3가 같은 값인 이유는 targets_3의 -100이 무시되었기 때문이다.  
왜 그럼 -100에서 하필 Cross-Entropy loss 계산이 제외된걸까?? 왜냐면 Pytorch의 cross_entropy 함수의 기본 설정에서 cross_entropy(..., ignore_index=-100)으로  
정해져 있기 때문이다. 즉, -100인 타깃은 무시한다는 의미이다. ignore_index를 사용해서 배치에 있는 훈련 샘플의 길이를 동일하게 만들기 위해 추가했던 텍스트 종료 토큰  
도 무시할 수 있다. 하지만 LLM이 응답의 끝을 나타내는 텍스트 종료 토큰을 생성하는 방법을 학습할 수 있도록 타깃에 하나의 50256 토큰을 남겨 둬야 한다.  
패딩 토큰을 마스킹하는 것 외에도 지시에 해당하는 타깃 토큰ID를 마스킹하는 것이 일반적이다. 지시에 해당하는 타깃 토큰 ID를 마스킹함으로서 생성된 응답 타깃ID에서만  
크로스 엔트로피 손실을 계산한다. 따라서 모델이 지시를 암기하지 않고 정확한 응답을 생성하는데 초점을 맞춰 훈련되므로 과대적합을 줄이는 데 도움이 된다.  
그러나 현재, 지시 토큰을 마스킹하는 것이 지시 미세 튜닝에 일반적으로 유용한지에 대해서는 연구자들의 의견이 갈린다. 

#### 7.4 지시 데이터셋을 위한 데이터 로더 만들기  
앞서 지시 데이터셋을 위한 여러 단계를 거쳐 InstructionDataset 클래스와 custom_collate_fn 함수를 구현했다. 
이 데이터 로더는 LLM 지시 미세 튜닝을 위해 자동으로 데이터를 섞고 배치를 만든다.  

In [63]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("장치:", device)

장치: cuda


In [64]:
from functools import partial

customized_collate_fn = partial(
    custom_collate_fn,
    device=device,
    allowed_max_length = 1024
)

In [65]:
from torch.utils.data import DataLoader

num_workers = 0
batch_size = 8

torch.manual_seed(123)

train_dataset = InstructionDataset(train_data, tokenizer)
train_loader = DataLoader(
    train_dataset,
    batch_size=batch_size,
    collate_fn = customized_collate_fn,
    shuffle=True,
    drop_last=True,
    num_workers=num_workers
)

val_dataset = InstructionDataset(val_data, tokenizer)
val_loader = DataLoader(
    val_dataset,
    batch_size=batch_size,
    collate_fn=customized_collate_fn,
    shuffle=False,
    drop_last=False,
    num_workers=num_workers
)

test_dataset = InstructionDataset(test_data, tokenizer)
test_loader = DataLoader(
    test_dataset,
    batch_size=batch_size,
    collate_fn=customized_collate_fn,
    shuffle=False,
    drop_last=False,
    num_workers=num_workers
)

In [66]:
print("훈련 데이터 로더:") 
for inputs, targets in train_loader:
    print(inputs.shape, targets.shape)

훈련 데이터 로더:
torch.Size([8, 61]) torch.Size([8, 61])
torch.Size([8, 76]) torch.Size([8, 76])
torch.Size([8, 73]) torch.Size([8, 73])
torch.Size([8, 68]) torch.Size([8, 68])
torch.Size([8, 65]) torch.Size([8, 65])
torch.Size([8, 72]) torch.Size([8, 72])
torch.Size([8, 80]) torch.Size([8, 80])
torch.Size([8, 67]) torch.Size([8, 67])
torch.Size([8, 62]) torch.Size([8, 62])
torch.Size([8, 75]) torch.Size([8, 75])
torch.Size([8, 62]) torch.Size([8, 62])
torch.Size([8, 68]) torch.Size([8, 68])
torch.Size([8, 67]) torch.Size([8, 67])
torch.Size([8, 77]) torch.Size([8, 77])
torch.Size([8, 69]) torch.Size([8, 69])
torch.Size([8, 79]) torch.Size([8, 79])
torch.Size([8, 71]) torch.Size([8, 71])
torch.Size([8, 66]) torch.Size([8, 66])
torch.Size([8, 83]) torch.Size([8, 83])
torch.Size([8, 68]) torch.Size([8, 68])
torch.Size([8, 80]) torch.Size([8, 80])
torch.Size([8, 71]) torch.Size([8, 71])
torch.Size([8, 69]) torch.Size([8, 69])
torch.Size([8, 65]) torch.Size([8, 65])
torch.Size([8, 68]) torch.Siz

#### 7.5 사전 훈련된 LLM 로드하기  
드디어 지도 학습 미세 튜닝 과정의 핵심 부분임. 나머지는 사전 훈련과 동일하므로 이전에 배웠던 코드를 재사용하면 된다!  
일단 지시 미세 튜닝을 시작하기 전에 미세 튜닝할 사전 훈련된 GPT 모델을 로드해야 한다. 이번에는 1억 2400만 파라미터 모델이 아니라 3억 5500만 파라미터의 중간 크기 모델을 로드한다!  
왜냐면 지시 미세 튜닝으로 만족스러운 결과를 내기 위해서는 기존 용량이 제한적이기 때문임!